# 01｜稳定去杠杆 ROE

研究范围：中证800动态成分股  
研究期间：2016-01-04 至 2025-12-31  
信号规则：T 日收盘后形成，T+1 日开盘交易  
样本划分：2016-2022 为设计期；2023-2025 为样本外期

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rqdatac import *
from rqfactor import *
from rqfactor import Factor

init()

# ---------- 固定研究配置 ----------
PROJECT_DIR = Path("./a-share-profitability-factor-replication")

INDEX_ID = "000906.XSHG"       # 中证800
START_DATE = "2016-01-04"
END_DATE = "2025-12-31"

RESEARCH_CONFIG = {
    "universe": "CSI 800",
    "index_id": INDEX_ID,
    "start_date": START_DATE,
    "end_date": END_DATE,
    "design_period": "2016-01-04 至 2022-12-31",
    "out_of_sample_period": "2023-01-01 至 2025-12-31",
    "signal_timing": "T日收盘后形成",
    "execution_timing": "T+1日开盘",
    "forward_return_days": 20,
}

# ---------- 动态中证800券池 ----------
def build_dynamic_index_universe(index_id, start_date, end_date):
    component_map = index_components(
        index_id,
        start_date=start_date,
        end_date=end_date,
    )

    index_fix = pd.DataFrame({
        date: dict.fromkeys(stocks, True)
        for date, stocks in component_map.items()
    }).T

    index_fix.index = pd.to_datetime(index_fix.index)
    index_fix = index_fix.sort_index()

    trading_dates = pd.to_datetime(
        get_trading_dates(start_date, end_date)
    )

    # 若指数成分只在调整日返回，则向后填充至每一个交易日
    index_fix = (
        index_fix
        .reindex(trading_dates)
        .ffill()
        .fillna(False)
        .astype(bool)
    )

    index_fix.index.name = "date"
    return index_fix

index_fix = build_dynamic_index_universe(
    INDEX_ID,
    START_DATE,
    END_DATE,
)

# 只保存“每日股票数”，不保存成分股明细
universe_summary = pd.DataFrame({
    "date": index_fix.index,
    "constituent_count": index_fix.sum(axis=1).astype(int),
})

# ---------- 保存公开结果 ----------
table_path = (
    PROJECT_DIR
    / "artifacts"
    / "tables"
    / "01_universe_summary.csv"
)

config_path = (
    PROJECT_DIR
    / "artifacts"
    / "metadata"
    / "01_research_config.json"
)

figure_path = (
    PROJECT_DIR
    / "artifacts"
    / "figures"
    / "01_csi800_universe_size.png"
)

universe_summary.to_csv(
    table_path,
    index=False,
    encoding="utf-8-sig",
)

config_path.write_text(
    json.dumps(RESEARCH_CONFIG, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(
    universe_summary["date"],
    universe_summary["constituent_count"],
    linewidth=1,
)
ax.set_title("CSI 800 Dynamic Universe Size")
ax.set_xlabel("Date")
ax.set_ylabel("Constituent Count")
ax.grid(alpha=0.25)
plt.tight_layout()
fig.savefig(figure_path, dpi=160, bbox_inches="tight")
plt.show()

print(f"券池日期数：{len(index_fix)}")
print(
    f"成分股数量范围："
    f"{universe_summary['constituent_count'].min()} - "
    f"{universe_summary['constituent_count'].max()}"
)
print(f"已保存：{table_path.resolve()}")
print(f"已保存：{config_path.resolve()}")
print(f"已保存：{figure_path.resolve()}")

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt

PROJECT_NAME = "a-share-profitability-factor-replication"
current_dir = Path.cwd().resolve()

# 自动识别项目根目录：兼容 Notebook 位于根目录或 notebooks 子目录两种情况
PROJECT_DIR = next(
    (path for path in [current_dir, *current_dir.parents]
     if path.name == PROJECT_NAME),
    None,
)

if PROJECT_DIR is None:
    PROJECT_DIR = next(
        (
            path / PROJECT_NAME
            for path in [current_dir, *current_dir.parents]
            if (path / PROJECT_NAME).is_dir()
        ),
        None,
    )

if PROJECT_DIR is None:
    raise FileNotFoundError(
        f"未找到 {PROJECT_NAME}。当前工作目录为：{current_dir}"
    )

# 若个别子目录不存在，仅补齐项目内部的标准目录
for relative_path in [
    "artifacts/tables",
    "artifacts/figures",
    "artifacts/metadata",
]:
    (PROJECT_DIR / relative_path).mkdir(parents=True, exist_ok=True)

table_path = PROJECT_DIR / "artifacts" / "tables" / "01_universe_summary.csv"
config_path = PROJECT_DIR / "artifacts" / "metadata" / "01_research_config.json"
figure_path = PROJECT_DIR / "artifacts" / "figures" / "01_csi800_universe_size.png"

# 保存公开聚合结果
universe_summary.to_csv(
    table_path,
    index=False,
    encoding="utf-8-sig",
)

config_path.write_text(
    json.dumps(RESEARCH_CONFIG, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(
    universe_summary["date"],
    universe_summary["constituent_count"],
    linewidth=1,
)
ax.set_title("CSI 800 Dynamic Universe Size")
ax.set_xlabel("Date")
ax.set_ylabel("Constituent Count")
ax.grid(alpha=0.25)
plt.tight_layout()
fig.savefig(figure_path, dpi=160, bbox_inches="tight")
plt.show()

print(f"当前 Notebook 工作目录：{current_dir}")
print(f"项目根目录：{PROJECT_DIR}")
print(f"券池日期数：{len(index_fix)}")
print(
    f"成分股数量范围："
    f"{universe_summary['constituent_count'].min()} - "
    f"{universe_summary['constituent_count'].max()}"
)
print(f"已保存：{table_path}")
print(f"已保存：{config_path}")
print(f"已保存：{figure_path}")

In [ ]:
from pathlib import Path
import json

import pandas as pd
import matplotlib.pyplot as plt

# 当前目录：
# .../a-share-profitability-factor-replication/a-share-profitability-factor-replication/notebooks
current_dir = Path.cwd().resolve()

# 正确项目根目录：向上两层
PROJECT_DIR = current_dir.parents[1]

if PROJECT_DIR.name != "a-share-profitability-factor-replication":
    raise ValueError(f"项目根目录识别异常：{PROJECT_DIR}")

for relative_path in [
    "artifacts/tables",
    "artifacts/figures",
    "artifacts/metadata",
]:
    (PROJECT_DIR / relative_path).mkdir(parents=True, exist_ok=True)

INDEX_ID = "000906.XSHG"
START_DATE = "2016-01-04"
END_DATE = "2025-12-31"

RESEARCH_CONFIG = {
    "universe": "CSI 800",
    "index_id": INDEX_ID,
    "start_date": START_DATE,
    "end_date": END_DATE,
    "design_period": "2016-01-04 至 2022-12-31",
    "out_of_sample_period": "2023-01-01 至 2025-12-31",
    "signal_timing": "T日收盘后形成",
    "execution_timing": "T+1日开盘",
    "forward_return_days": 20,
}

def build_dynamic_index_universe(index_id, start_date, end_date):
    component_map = index_components(
        index_id,
        start_date=start_date,
        end_date=end_date,
    )

    # 每个“实际返回的成分调整日”先独立补 False；
    # 必须在扩展到每日交易日以前完成，防止旧成分被错误延续。
    membership_on_change_dates = pd.DataFrame({
        date: dict.fromkeys(stocks, True)
        for date, stocks in component_map.items()
    }).T

    membership_on_change_dates.index = pd.to_datetime(
        membership_on_change_dates.index
    )

    membership_on_change_dates = (
        membership_on_change_dates
        .sort_index()
        .fillna(False)
        .astype(bool)
    )

    trading_dates = pd.to_datetime(
        get_trading_dates(start_date, end_date)
    )

    # 仅把“已经正确记录的整期成员关系”延续到非调整日
    return (
        membership_on_change_dates
        .reindex(trading_dates)
        .ffill()
        .fillna(False)
        .astype(bool)
    )

index_fix = build_dynamic_index_universe(
    INDEX_ID,
    START_DATE,
    END_DATE,
)

universe_summary = pd.DataFrame({
    "date": index_fix.index,
    "constituent_count": index_fix.sum(axis=1).astype(int),
})

# 中证800在本研究区间内应始终为 800 只
if not (universe_summary["constituent_count"] == 800).all():
    unexpected = universe_summary[
        universe_summary["constituent_count"] != 800
    ]
    raise ValueError(
        "券池数量仍非800，请勿继续。异常示例：\n"
        f"{unexpected.head()}"
    )

table_path = PROJECT_DIR / "artifacts/tables/01_universe_summary.csv"
config_path = PROJECT_DIR / "artifacts/metadata/01_research_config.json"
figure_path = PROJECT_DIR / "artifacts/figures/01_csi800_universe_size.png"

universe_summary.to_csv(
    table_path,
    index=False,
    encoding="utf-8-sig",
)

config_path.write_text(
    json.dumps(RESEARCH_CONFIG, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(
    universe_summary["date"],
    universe_summary["constituent_count"],
    linewidth=1,
)
ax.set_title("CSI 800 Dynamic Universe Size")
ax.set_xlabel("Date")
ax.set_ylabel("Constituent Count")
ax.grid(alpha=0.25)
plt.tight_layout()
fig.savefig(figure_path, dpi=160, bbox_inches="tight")
plt.show()

print(f"正确项目根目录：{PROJECT_DIR}")
print(f"券池日期数：{len(index_fix)}")
print(
    f"成分股数量范围："
    f"{universe_summary['constituent_count'].min()} - "
    f"{universe_summary['constituent_count'].max()}"
)
print(f"已保存：{table_path}")
print(f"已保存：{config_path}")
print(f"已保存：{figure_path}")

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 取最外层项目目录，避开当前的重复嵌套目录
PROJECT_NAME = "a-share-profitability-factor-replication"
all_project_dirs = [
    path
    for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if path.name == PROJECT_NAME
]
PROJECT_DIR = all_project_dirs[-1]

# ---------- 每月最后一个交易日生成信号 ----------
signal_dates = pd.DatetimeIndex(
    pd.Series(
        index_fix.index,
        index=index_fix.index.to_period("M"),
    ).groupby(level=0).max().to_list()
)

stock_list = index_fix.columns.tolist()

# 月末动态中证800成员
index_membership = (
    index_fix
    .reindex(index=signal_dates, columns=stock_list)
    .fillna(False)
    .astype(bool)
)

# ---------- T日已知的新股过滤 ----------
listing_dates = {
    stock: pd.Timestamp(instruments(stock).listed_date)
    for stock in stock_list
}

seasoned_dates = {
    stock: pd.Timestamp(
        get_next_trading_date(listed_date, n=252)
    )
    for stock, listed_date in listing_dates.items()
}

new_stock_filter = pd.DataFrame(
    {
        stock: signal_dates < seasoned_date
        for stock, seasoned_date in seasoned_dates.items()
    },
    index=signal_dates,
)

# ---------- T日已知的ST、停牌状态 ----------
st_filter = (
    is_st_stock(
        stock_list,
        signal_dates[0],
        signal_dates[-1],
    )
    .reindex(index=signal_dates, columns=stock_list)
    .fillna(False)
    .astype(bool)
)

suspended_filter = (
    is_suspended(
        stock_list,
        signal_dates[0],
        signal_dates[-1],
    )
    .reindex(index=signal_dates, columns=stock_list)
    .fillna(False)
    .astype(bool)
)

# 研究层资格掩码：绝不使用任何 T+1 信息
research_mask = (
    index_membership
    & (~new_stock_filter)
    & (~st_filter)
    & (~suspended_filter)
)

# ---------- 只保存日期级统计，不保存股票级掩码 ----------
mask_summary = pd.DataFrame({
    "date": signal_dates,
    "index_members": index_membership.sum(axis=1).astype(int),
    "new_stock_excluded": (
        index_membership & new_stock_filter
    ).sum(axis=1).astype(int),
    "st_excluded": (
        index_membership & st_filter
    ).sum(axis=1).astype(int),
    "suspended_excluded": (
        index_membership & suspended_filter
    ).sum(axis=1).astype(int),
    "eligible_stocks": research_mask.sum(axis=1).astype(int),
})

mask_rules = {
    "signal_frequency": "每月最后一个交易日",
    "universe": "中证800动态成分股",
    "new_stock_rule": "上市满252个交易日",
    "st_rule": "仅剔除T日ST股票",
    "suspension_rule": "仅剔除T日停牌股票",
    "future_information_rule": "不使用T+1涨停、停牌、ST信息",
    "execution_note": "T+1能否成交只在最终策略回测层处理",
}

table_path = (
    PROJECT_DIR
    / "artifacts"
    / "tables"
    / "01_research_mask_summary.csv"
)

metadata_path = (
    PROJECT_DIR
    / "artifacts"
    / "metadata"
    / "01_research_mask_rules.json"
)

figure_path = (
    PROJECT_DIR
    / "artifacts"
    / "figures"
    / "01_eligible_stock_count.png"
)

mask_summary.to_csv(
    table_path,
    index=False,
    encoding="utf-8-sig",
)

metadata_path.write_text(
    json.dumps(mask_rules, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(
    mask_summary["date"],
    mask_summary["eligible_stocks"],
    linewidth=1,
)
ax.set_title("Monthly Eligible Stocks After T-date Filters")
ax.set_xlabel("Date")
ax.set_ylabel("Eligible Stock Count")
ax.grid(alpha=0.25)
plt.tight_layout()
fig.savefig(figure_path, dpi=160, bbox_inches="tight")
plt.show()

print(f"信号日期数：{len(signal_dates)}")
print(
    f"可研究股票数量范围："
    f"{mask_summary['eligible_stocks'].min()} - "
    f"{mask_summary['eligible_stocks'].max()}"
)
print(f"项目根目录：{PROJECT_DIR}")
print(f"已保存：{table_path}")
print(f"已保存：{metadata_path}")
print(f"已保存：{figure_path}")

## 原始输入提取与覆盖率审计

提取 ROE、9期历史 TTM ROE、总资产和总权益。
原始因子面板仅保留在米筐内存；对外仅保存日期级覆盖率统计。

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_NAME = "a-share-profitability-factor-replication"
all_project_dirs = [
    path
    for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if path.name == PROJECT_NAME
]
PROJECT_DIR = all_project_dirs[-1]

# 所有字段均在 00 阶段完成字段审计
input_fields = {
    "roe_ttm_baseline": "return_on_equity_ttm",
    "roe_ttm_lag_0": "return_on_equity_weighted_average_ttm_0",
    "roe_ttm_lag_1": "return_on_equity_weighted_average_ttm_1",
    "roe_ttm_lag_2": "return_on_equity_weighted_average_ttm_2",
    "roe_ttm_lag_3": "return_on_equity_weighted_average_ttm_3",
    "roe_ttm_lag_4": "return_on_equity_weighted_average_ttm_4",
    "roe_ttm_lag_5": "return_on_equity_weighted_average_ttm_5",
    "roe_ttm_lag_6": "return_on_equity_weighted_average_ttm_6",
    "roe_ttm_lag_7": "return_on_equity_weighted_average_ttm_7",
    "roe_ttm_lag_8": "return_on_equity_weighted_average_ttm_8",
    "total_assets_mrq": "total_assets_mrq_0",
    "total_equity_mrq": "total_equity_mrq_0",
}

def fetch_monthly_factor(field_name):
    panel = execute_factor(
        Factor(field_name),
        stock_list,
        START_DATE,
        END_DATE,
    )

    panel.index = pd.to_datetime(panel.index)

    return (
        panel
        .reindex(index=signal_dates, columns=stock_list)
        .astype(float)
    )

# 原始面板仅驻留内存，不写入本地文件
factor_inputs = {}

for label, field_name in input_fields.items():
    print(f"正在提取：{label}")
    factor_inputs[label] = fetch_monthly_factor(field_name)

roe_ttm_baseline_raw = factor_inputs["roe_ttm_baseline"]
roe_lag_panels_raw = [
    factor_inputs[f"roe_ttm_lag_{lag}"]
    for lag in range(9)
]
total_assets_raw = factor_inputs["total_assets_mrq"]
total_equity_raw = factor_inputs["total_equity_mrq"]

# 研究掩码只在这里生效；不使用 T+1 信息
masked_inputs = {
    name: panel.where(research_mask)
    for name, panel in factor_inputs.items()
}

# 仅保存日期级有效观测数
coverage_summary = pd.DataFrame(index=signal_dates)
coverage_summary.index.name = "date"

for name, panel in masked_inputs.items():
    coverage_summary[name] = panel.notna().sum(axis=1).astype(int)

coverage_summary["eligible_stocks"] = (
    research_mask.sum(axis=1).astype(int)
)

coverage_summary = coverage_summary.reset_index()

table_path = (
    PROJECT_DIR
    / "artifacts"
    / "tables"
    / "01_raw_input_coverage.csv"
)

metadata_path = (
    PROJECT_DIR
    / "artifacts"
    / "metadata"
    / "01_input_fields.json"
)

figure_path = (
    PROJECT_DIR
    / "artifacts"
    / "figures"
    / "01_raw_input_coverage.png"
)

coverage_summary.to_csv(
    table_path,
    index=False,
    encoding="utf-8-sig",
)

metadata_path.write_text(
    json.dumps(input_fields, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

fig, ax = plt.subplots(figsize=(11, 4))

for column in [
    "roe_ttm_lag_0",
    "roe_ttm_lag_8",
    "total_assets_mrq",
    "total_equity_mrq",
]:
    ax.plot(
        coverage_summary["date"],
        coverage_summary[column],
        label=column,
        linewidth=1,
    )

ax.plot(
    coverage_summary["date"],
    coverage_summary["eligible_stocks"],
    label="eligible_stocks",
    color="black",
    linewidth=1.5,
    linestyle="--",
)

ax.set_title("Stable ROE Input Coverage")
ax.set_xlabel("Date")
ax.set_ylabel("Valid Stock Count")
ax.legend(ncol=2)
ax.grid(alpha=0.25)
plt.tight_layout()
fig.savefig(figure_path, dpi=160, bbox_inches="tight")
plt.show()

print(f"输入字段数：{len(input_fields)}")
print(
    "当前ROE平均有效数："
    f"{coverage_summary['roe_ttm_lag_0'].mean():.1f}"
)
print(
    "8期滞后ROE平均有效数："
    f"{coverage_summary['roe_ttm_lag_8'].mean():.1f}"
)
print(f"已保存：{table_path}")
print(f"已保存：{metadata_path}")
print(f"已保存：{figure_path}")

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_NAME = "a-share-profitability-factor-replication"
all_project_dirs = [
    path
    for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if path.name == PROJECT_NAME
]
PROJECT_DIR = all_project_dirs[-1]

# ---------- 仅使用通过 T 日筛选的股票 ----------
baseline_roe_raw = roe_ttm_baseline_raw.where(research_mask)

roe_lag_panels = [
    panel.where(research_mask)
    for panel in roe_lag_panels_raw
]

assets = total_assets_raw.where(research_mask)
equity = total_equity_raw.where(research_mask)

# 负权益或非正资产下，权益乘数没有合理经济解释
equity_multiplier_raw = (
    assets / equity
).where((assets > 0) & (equity > 0))


# ---------- 截面工具函数 ----------
def mad_winsorize(series, n=3.5):
    series = series.astype(float).copy()

    median = series.median()
    mad = (series - median).abs().median()

    if pd.isna(mad) or mad == 0:
        return series

    lower = median - n * 1.4826 * mad
    upper = median + n * 1.4826 * mad
    return series.clip(lower, upper)


def cross_section_zscore(series):
    series = series.astype(float).copy()
    std = series.std(ddof=0)

    if pd.isna(std) or std == 0:
        return series * np.nan

    return (series - series.mean()) / std


def remove_leverage_effect(roe_panel, leverage_panel, min_obs=50):
    """
    每月横截面回归：
    ROE = alpha + beta × equity_multiplier + residual
    residual 即去杠杆 ROE。
    """
    result = pd.DataFrame(
        np.nan,
        index=roe_panel.index,
        columns=roe_panel.columns,
    )

    for date in roe_panel.index:
        cross_section = pd.DataFrame({
            "roe": roe_panel.loc[date],
            "leverage": leverage_panel.loc[date],
        }).dropna()

        if len(cross_section) < min_obs:
            continue

        y = mad_winsorize(cross_section["roe"])
        x = mad_winsorize(cross_section["leverage"])

        valid = pd.concat([y, x], axis=1).dropna()

        if len(valid) < min_obs:
            continue

        X = np.column_stack([
            np.ones(len(valid)),
            valid["leverage"].to_numpy(),
        ])

        beta = np.linalg.lstsq(
            X,
            valid["roe"].to_numpy(),
            rcond=None,
        )[0]

        residual = valid["roe"].to_numpy() - X @ beta
        result.loc[date, valid.index] = residual

    return result


# ---------- 1. 去杠杆 ROE ----------
roe_current = roe_lag_panels[0]

delevered_roe_raw = remove_leverage_effect(
    roe_current,
    equity_multiplier_raw,
)

# ---------- 2. 9期 ROE 稳定性 ----------
roe_lag_stack = np.stack(
    [panel.to_numpy(dtype=float) for panel in roe_lag_panels],
    axis=0,
)

all_nine_periods_available = np.isfinite(
    roe_lag_stack
).all(axis=0)

roe_volatility_array = np.full(
    all_nine_periods_available.shape,
    np.nan,
)

roe_volatility_array[all_nine_periods_available] = np.std(
    roe_lag_stack[:, all_nine_periods_available],
    axis=0,
    ddof=1,
)

roe_volatility_raw = pd.DataFrame(
    roe_volatility_array,
    index=signal_dates,
    columns=stock_list,
)

# ---------- 3. 融合：高去杠杆ROE + 低ROE波动 ----------
delevered_roe_z = delevered_roe_raw.apply(
    cross_section_zscore,
    axis=1,
)

roe_stability_z = (-roe_volatility_raw).apply(
    cross_section_zscore,
    axis=1,
)

stable_delevered_roe_raw = (
    0.5 * delevered_roe_z
    + 0.5 * roe_stability_z
).where(research_mask)

# ---------- 只保存日期级覆盖率 ----------
factor_summary = pd.DataFrame({
    "date": signal_dates,
    "eligible_stocks": research_mask.sum(axis=1).astype(int),
    "baseline_roe_valid": baseline_roe_raw.notna().sum(axis=1).astype(int),
    "equity_multiplier_valid": equity_multiplier_raw.notna().sum(axis=1).astype(int),
    "delevered_roe_valid": delevered_roe_raw.notna().sum(axis=1).astype(int),
    "roe_volatility_valid": roe_volatility_raw.notna().sum(axis=1).astype(int),
    "stable_delevered_roe_valid": (
        stable_delevered_roe_raw.notna().sum(axis=1).astype(int)
    ),
})

rules = {
    "equity_multiplier": "total_assets_mrq_0 / total_equity_mrq_0",
    "negative_equity_treatment": "剔除总权益小于等于0的股票",
    "delevered_roe": "每月横截面回归ROE对权益乘数，使用残差",
    "roe_stability": "9期TTM ROE的样本标准差",
    "stable_delevered_roe": (
        "0.5*zscore(去杠杆ROE) + 0.5*zscore(-ROE波动率)"
    ),
    "missing_value_rule": "9期ROE未完整披露则不计算稳定性，不填补",
    "neutralization_status": "尚未进行行业与市值中性化",
}

table_path = (
    PROJECT_DIR
    / "artifacts"
    / "tables"
    / "01_pre_neutralization_factor_summary.csv"
)

metadata_path = (
    PROJECT_DIR
    / "artifacts"
    / "metadata"
    / "01_factor_construction_rules.json"
)

figure_path = (
    PROJECT_DIR
    / "artifacts"
    / "figures"
    / "01_factor_construction_coverage.png"
)

factor_summary.to_csv(
    table_path,
    index=False,
    encoding="utf-8-sig",
)

metadata_path.write_text(
    json.dumps(rules, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

fig, ax = plt.subplots(figsize=(11, 4))

for column in [
    "eligible_stocks",
    "delevered_roe_valid",
    "roe_volatility_valid",
    "stable_delevered_roe_valid",
]:
    ax.plot(
        factor_summary["date"],
        factor_summary[column],
        label=column,
        linewidth=1.1,
    )

ax.set_title("Stable Delevered ROE Construction Coverage")
ax.set_xlabel("Date")
ax.set_ylabel("Valid Stock Count")
ax.legend(ncol=2)
ax.grid(alpha=0.25)
plt.tight_layout()
fig.savefig(figure_path, dpi=160, bbox_inches="tight")
plt.show()

print(
    "稳定去杠杆ROE平均有效股票数："
    f"{factor_summary['stable_delevered_roe_valid'].mean():.1f}"
)
print(
    "稳定去杠杆ROE最小有效股票数："
    f"{factor_summary['stable_delevered_roe_valid'].min()}"
)
print(f"已保存：{table_path}")
print(f"已保存：{metadata_path}")
print(f"已保存：{figure_path}")

In [ ]:
# ============================================================
# 01.4 中性化输入：历史中信行业 + 当期市值
# 只保存覆盖率汇总，不保存原始市值或行业面板
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import rqdatac
from rqdatac import get_trading_dates, get_previous_trading_date
from rqfactor import Factor, execute_factor

# ---------- 1. 定位最外层项目目录 ----------
PROJECT_NAME = "a-share-profitability-factor-replication"

project_candidates = [
    path
    for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if path.name == PROJECT_NAME
]
if not project_candidates:
    raise RuntimeError("未找到项目根目录，请确认当前 Notebook 在项目文件夹中。")

PROJECT_DIR = project_candidates[-1]   # 取最外层同名目录
TABLE_DIR = PROJECT_DIR / "artifacts" / "tables"
META_DIR = PROJECT_DIR / "artifacts" / "metadata"
FIG_DIR = PROJECT_DIR / "artifacts" / "figures"

for directory in [TABLE_DIR, META_DIR, FIG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# ---------- 2. 检查前面步骤产生的变量 ----------
required_variables = [
    "stock_list",
    "signal_dates",
    "research_mask",
    "stable_delevered_roe_raw",
    "START_DATE",
    "END_DATE",
]
missing_variables = [name for name in required_variables if name not in globals()]
if missing_variables:
    raise RuntimeError(
        "请先依次运行本 Notebook 前面的单元格，缺少变量："
        + ", ".join(missing_variables)
    )

signal_dates = pd.DatetimeIndex(pd.to_datetime(signal_dates))
research_mask = research_mask.reindex(index=signal_dates, columns=stock_list).fillna(False)

# ---------- 3. 提取信号日可得的流通市值 ----------
market_cap_raw = execute_factor(
    Factor("market_cap_3"),
    stock_list,
    START_DATE,
    END_DATE,
)
market_cap_raw.index = pd.to_datetime(market_cap_raw.index)

market_cap_raw = (
    market_cap_raw
    .reindex(index=signal_dates, columns=stock_list)
    .astype(float)
    .where(research_mask)
)

# 中性化使用 log(市值)，非正市值视为缺失
log_market_cap_raw = np.log(market_cap_raw.where(market_cap_raw > 0))


# ---------- 4. 提取历史中信一级行业；只向前填充，绝不向未来取值 ----------
def get_zx_2019_industry_panel(order_book_ids, target_dates):
    """
    返回：index=信号日、columns=股票、value=中信一级行业名称。
    行业变更仅从生效日起向后使用，不使用未来行业标签。
    """
    target_dates = pd.DatetimeIndex(pd.to_datetime(target_dates))

    raw_industry_history = rqdatac.client.get_client().execute(
        "__internal__zx2019_industry"
    )

    industry_history = pd.DataFrame(
        raw_industry_history,
        columns=["first_industry_name", "order_book_id", "start_date"],
    )

    industry_history["start_date"] = pd.to_datetime(industry_history["start_date"])
    industry_history = industry_history.dropna(
        subset=["first_industry_name", "order_book_id", "start_date"]
    )
    industry_history = industry_history[
        industry_history["order_book_id"].isin(order_book_ids)
    ].sort_values(["start_date", "order_book_id"])

    industry_changes = industry_history.pivot_table(
        index="start_date",
        columns="order_book_id",
        values="first_industry_name",
        aggfunc="last",
    ).sort_index()

    calendar_start = get_previous_trading_date(target_dates.min(), 2)
    trading_calendar = pd.DatetimeIndex(
        pd.to_datetime(
            get_trading_dates(calendar_start, target_dates.max())
        )
    )

    # 分类仅从历史生效日期向后填充
    industry_panel = (
        industry_changes
        .reindex(trading_calendar)
        .ffill()
        .reindex(target_dates)
        .ffill()
        .reindex(columns=order_book_ids)
    )
    return industry_panel


zx_industry_panel = get_zx_2019_industry_panel(stock_list, signal_dates)
zx_industry_panel = zx_industry_panel.where(research_mask)

# ---------- 5. 覆盖率检查 ----------
eligible_count = research_mask.sum(axis=1)
stable_factor_count = stable_delevered_roe_raw.notna().sum(axis=1)
log_market_cap_count = log_market_cap_raw.notna().sum(axis=1)
industry_count = zx_industry_panel.notna().sum(axis=1)

neutralization_ready_mask = (
    stable_delevered_roe_raw.notna()
    & log_market_cap_raw.notna()
    & zx_industry_panel.notna()
    & research_mask
)
neutralization_ready_count = neutralization_ready_mask.sum(axis=1)

neutralization_coverage = pd.DataFrame({
    "eligible_stocks": eligible_count,
    "stable_delevered_roe_valid": stable_factor_count,
    "log_market_cap_valid": log_market_cap_count,
    "zx_industry_valid": industry_count,
    "ready_for_neutralization": neutralization_ready_count,
})
neutralization_coverage.index.name = "signal_date"

# ---------- 6. 保存可开源的汇总结果 ----------
table_path = TABLE_DIR / "01_neutralization_input_coverage.csv"
neutralization_coverage.to_csv(
    table_path,
    encoding="utf-8-sig",
)

metadata = {
    "market_cap_field": "market_cap_3",
    "market_cap_transform": "log(market_cap_3), with non-positive values set to missing",
    "industry_classification": "CITIC ZX 2019 first-level industry",
    "industry_timing_rule": "industry labels are forward-filled only from each historical effective date",
    "neutralization_inputs": [
        "stable_delevered_roe_raw",
        "log_market_cap_raw",
        "zx_industry_panel",
    ],
    "raw_vendor_data_saved_to_repository": False,
    "research_date_rule": "all inputs are aligned to the monthly signal date T",
}
metadata_path = META_DIR / "01_neutralization_input_rules.json"
with open(metadata_path, "w", encoding="utf-8") as file:
    json.dump(metadata, file, ensure_ascii=False, indent=2)

# ---------- 7. 保存覆盖率图 ----------
plt.figure(figsize=(12, 5))
for column in neutralization_coverage.columns:
    plt.plot(
        neutralization_coverage.index,
        neutralization_coverage[column],
        label=column,
        linewidth=1.5,
    )

plt.title("Neutralization Input Coverage")
plt.xlabel("Date")
plt.ylabel("Stock Count")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()

figure_path = FIG_DIR / "01_neutralization_input_coverage.png"
plt.savefig(figure_path, dpi=160, bbox_inches="tight")
plt.show()

print("项目根目录：", PROJECT_DIR)
print("市值平均有效股票数：{:.1f}".format(log_market_cap_count.mean()))
print("行业平均有效股票数：{:.1f}".format(industry_count.mean()))
print("可进入中性化的平均股票数：{:.1f}".format(neutralization_ready_count.mean()))
print("可进入中性化的最小股票数：{}".format(neutralization_ready_count.min()))
print("已保存：", table_path)
print("已保存：", metadata_path)
print("已保存：", figure_path)

In [ ]:
# ============================================================
# 01.4A 行业分类来源诊断
# 比较 citics / citics_2019 / gildata 的历史覆盖率
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rqdatac

# ---------- 1. 项目目录 ----------
PROJECT_NAME = "a-share-profitability-factor-replication"
project_candidates = [
    path
    for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if path.name == PROJECT_NAME
]
PROJECT_DIR = project_candidates[-1]

TABLE_DIR = PROJECT_DIR / "artifacts" / "tables"
META_DIR = PROJECT_DIR / "artifacts" / "metadata"
FIG_DIR = PROJECT_DIR / "artifacts" / "figures"

for directory in [TABLE_DIR, META_DIR, FIG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# ---------- 2. 选择六个代表性信号日 ----------
check_years = [2016, 2017, 2019, 2020, 2023, 2025]
diagnostic_dates = []

for year in check_years:
    dates_in_year = signal_dates[signal_dates.year == year]
    if len(dates_in_year) > 0:
        diagnostic_dates.append(dates_in_year[-1])

industry_sources = ["citics", "citics_2019", "gildata"]


# ---------- 3. 单日、单来源覆盖率 ----------
def diagnose_industry_source(as_of_date, source):
    eligible_ids = research_mask.columns[
        research_mask.loc[as_of_date].fillna(False)
    ].tolist()

    try:
        raw = rqdatac.get_instrument_industry(
            order_book_ids=stock_list,
            source=source,
            level=1,
            date=pd.Timestamp(as_of_date),
            market="cn",
        )

        industry_df = raw.copy()

        # 兼容不同返回格式：股票代码可能在 index 或 column 中
        if "order_book_id" in industry_df.columns:
            industry_df = industry_df.set_index("order_book_id")

        industry_df.index = industry_df.index.astype(str)

        if "first_industry_name" not in industry_df.columns:
            raise ValueError(
                "返回结果中没有 first_industry_name 字段，实际字段为："
                + str(industry_df.columns.tolist())
            )

        valid_industry_count = (
            industry_df.reindex(eligible_ids)["first_industry_name"]
            .notna()
            .sum()
        )

        return {
            "signal_date": pd.Timestamp(as_of_date),
            "source": source,
            "eligible_stocks": len(eligible_ids),
            "industry_valid": int(valid_industry_count),
            "coverage_pct": round(
                100 * valid_industry_count / len(eligible_ids), 2
            ),
            "status": "ok",
            "error_message": "",
        }

    except Exception as error:
        return {
            "signal_date": pd.Timestamp(as_of_date),
            "source": source,
            "eligible_stocks": len(eligible_ids),
            "industry_valid": np.nan,
            "coverage_pct": np.nan,
            "status": "error",
            "error_message": str(error)[:300],
        }


# ---------- 4. 执行诊断 ----------
diagnostic_records = []

for as_of_date in diagnostic_dates:
    for source in industry_sources:
        result = diagnose_industry_source(as_of_date, source)
        diagnostic_records.append(result)
        print(
            "{} | {:12s} | {:>6} | 覆盖率：{}".format(
                pd.Timestamp(as_of_date).strftime("%Y-%m-%d"),
                source,
                result["status"],
                result["coverage_pct"],
            )
        )

industry_source_diagnosis = pd.DataFrame(diagnostic_records)
industry_source_diagnosis = industry_source_diagnosis.sort_values(
    ["signal_date", "source"]
)

# ---------- 5. 保存可开源的诊断汇总 ----------
table_path = TABLE_DIR / "01_industry_source_diagnosis.csv"
industry_source_diagnosis.to_csv(
    table_path,
    index=False,
    encoding="utf-8-sig",
)

metadata = {
    "purpose": "Compare historical industry classification coverage before factor neutralization",
    "sources_tested": industry_sources,
    "dates_tested": [
        pd.Timestamp(date).strftime("%Y-%m-%d")
        for date in diagnostic_dates
    ],
    "selection_rule": (
        "Select a source only if it has stable coverage close to 100% "
        "across all representative dates."
    ),
    "raw_vendor_industry_data_saved_to_repository": False,
}
metadata_path = META_DIR / "01_industry_source_diagnosis_rules.json"
with open(metadata_path, "w", encoding="utf-8") as file:
    json.dump(metadata, file, ensure_ascii=False, indent=2)

# ---------- 6. 作图 ----------
ok_result = industry_source_diagnosis[
    industry_source_diagnosis["status"] == "ok"
].copy()

coverage_plot = ok_result.pivot(
    index="signal_date",
    columns="source",
    values="coverage_pct",
)

plt.figure(figsize=(11, 5))
for source in coverage_plot.columns:
    plt.plot(
        coverage_plot.index,
        coverage_plot[source],
        marker="o",
        linewidth=1.8,
        label=source,
    )

plt.axhline(95, linestyle="--", color="gray", linewidth=1, label="95% threshold")
plt.title("Historical Industry Classification Coverage by Source")
plt.xlabel("Signal Date")
plt.ylabel("Coverage of Eligible Stocks (%)")
plt.ylim(0, 105)
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()

figure_path = FIG_DIR / "01_industry_source_diagnosis.png"
plt.savefig(figure_path, dpi=160, bbox_inches="tight")
plt.show()

print("\n===== 覆盖率汇总 =====")
print(
    industry_source_diagnosis.pivot(
        index="signal_date",
        columns="source",
        values="coverage_pct",
    )
)

print("\n已保存：", table_path)
print("已保存：", metadata_path)
print("已保存：", figure_path)

In [ ]:
# ============================================================
# 01.4B 重建完整历史中信 2019 一级行业面板
# 使用逐信号日 as-of 查询；不保存原始行业标签
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rqdatac

# ---------- 1. 前置变量检查 ----------
required_variables = [
    "stock_list",
    "signal_dates",
    "research_mask",
    "stable_delevered_roe_raw",
    "log_market_cap_raw",
]
missing_variables = [
    name for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "请先运行前面的单元格，缺少变量："
        + ", ".join(missing_variables)
    )

# ---------- 2. 项目目录 ----------
PROJECT_NAME = "a-share-profitability-factor-replication"
project_candidates = [
    path
    for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if path.name == PROJECT_NAME
]
PROJECT_DIR = project_candidates[-1]

TABLE_DIR = PROJECT_DIR / "artifacts" / "tables"
META_DIR = PROJECT_DIR / "artifacts" / "metadata"
FIG_DIR = PROJECT_DIR / "artifacts" / "figures"

for directory in [TABLE_DIR, META_DIR, FIG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

signal_dates = pd.DatetimeIndex(pd.to_datetime(signal_dates))
research_mask = research_mask.reindex(
    index=signal_dates,
    columns=stock_list,
).fillna(False)

# ---------- 3. 逐月获取 T 日可得的中信 2019 一级行业 ----------
SELECTED_INDUSTRY_SOURCE = "citics_2019"
INDUSTRY_FIELD = "first_industry_name"

industry_panel_raw = pd.DataFrame(
    index=signal_dates,
    columns=stock_list,
    dtype="object",
)

query_log = []

for sequence, as_of_date in enumerate(signal_dates, start=1):
    try:
        industry_df = rqdatac.get_instrument_industry(
            order_book_ids=stock_list,
            source=SELECTED_INDUSTRY_SOURCE,
            level=1,
            date=pd.Timestamp(as_of_date),
            market="cn",
        ).copy()

        # 兼容“股票代码在 index 或 column”两种返回格式
        if "order_book_id" in industry_df.columns:
            industry_df = industry_df.set_index("order_book_id")

        industry_df.index = industry_df.index.astype(str)

        if INDUSTRY_FIELD not in industry_df.columns:
            raise ValueError(
                "未找到行业字段 {}，实际字段：{}".format(
                    INDUSTRY_FIELD,
                    industry_df.columns.tolist(),
                )
            )

        industry_series = industry_df[INDUSTRY_FIELD].reindex(stock_list)
        industry_panel_raw.loc[as_of_date] = industry_series.values

        query_log.append({
            "signal_date": as_of_date,
            "returned_stock_count": int(industry_series.notna().sum()),
            "status": "ok",
            "error_message": "",
        })

    except Exception as error:
        query_log.append({
            "signal_date": as_of_date,
            "returned_stock_count": np.nan,
            "status": "error",
            "error_message": str(error)[:300],
        })
        raise RuntimeError(
            "行业数据在 {} 查询失败：{}".format(
                pd.Timestamp(as_of_date).strftime("%Y-%m-%d"),
                error,
            )
        )

    if sequence % 12 == 0 or sequence == len(signal_dates):
        print(
            "已完成 {}/{} 个信号日行业查询".format(
                sequence,
                len(signal_dates),
            )
        )

# ---------- 4. 仅保留当期可研究股票 ----------
industry_panel = industry_panel_raw.where(research_mask)

# 重要：覆盖旧变量，后续中性化只能使用修正后的面板
zx_industry_panel = industry_panel

# ---------- 5. 重算中性化覆盖率 ----------
eligible_count = research_mask.sum(axis=1)
stable_factor_count = stable_delevered_roe_raw.notna().sum(axis=1)
log_market_cap_count = log_market_cap_raw.notna().sum(axis=1)
industry_count = industry_panel.notna().sum(axis=1)

neutralization_ready_mask = (
    stable_delevered_roe_raw.notna()
    & log_market_cap_raw.notna()
    & industry_panel.notna()
    & research_mask
)

neutralization_ready_count = neutralization_ready_mask.sum(axis=1)
industry_group_count = industry_panel.nunique(axis=1)

neutralization_coverage = pd.DataFrame({
    "eligible_stocks": eligible_count,
    "stable_delevered_roe_valid": stable_factor_count,
    "log_market_cap_valid": log_market_cap_count,
    "citics_2019_industry_valid": industry_count,
    "ready_for_neutralization": neutralization_ready_count,
})
neutralization_coverage.index.name = "signal_date"

selected_industry_coverage = pd.DataFrame({
    "eligible_stocks": eligible_count,
    "citics_2019_industry_valid": industry_count,
    "industry_coverage_pct": (
        100 * industry_count / eligible_count
    ).round(2),
    "first_level_industry_group_count": industry_group_count,
})
selected_industry_coverage.index.name = "signal_date"

# ---------- 6. 保存可开源汇总 ----------
# 覆盖此前错误的行业覆盖率输出
coverage_path = TABLE_DIR / "01_neutralization_input_coverage.csv"
neutralization_coverage.to_csv(
    coverage_path,
    encoding="utf-8-sig",
)

industry_coverage_path = TABLE_DIR / "01_selected_industry_coverage.csv"
selected_industry_coverage.to_csv(
    industry_coverage_path,
    encoding="utf-8-sig",
)

query_log_path = TABLE_DIR / "01_industry_query_log.csv"
pd.DataFrame(query_log).to_csv(
    query_log_path,
    index=False,
    encoding="utf-8-sig",
)

metadata = {
    "selected_industry_source": "citics_2019",
    "industry_level": "first level",
    "industry_field": "first_industry_name",
    "historical_query_rule": (
        "For every monthly signal date T, query industry "
        "classification using date=T; no future label is used."
    ),
    "market_cap_field": "market_cap_3",
    "market_cap_transform": "log(market_cap_3), non-positive values treated as missing",
    "raw_vendor_industry_panel_saved_to_repository": False,
    "previous_internal_change_log_method": "rejected due to incomplete historical coverage",
}
metadata_path = META_DIR / "01_neutralization_input_rules.json"
with open(metadata_path, "w", encoding="utf-8") as file:
    json.dump(metadata, file, ensure_ascii=False, indent=2)

# ---------- 7. 保存修正后的覆盖率图 ----------
plt.figure(figsize=(12, 5))

for column in neutralization_coverage.columns:
    plt.plot(
        neutralization_coverage.index,
        neutralization_coverage[column],
        label=column,
        linewidth=1.5,
    )

plt.title("Corrected Neutralization Input Coverage")
plt.xlabel("Date")
plt.ylabel("Stock Count")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()

figure_path = FIG_DIR / "01_neutralization_input_coverage.png"
plt.savefig(figure_path, dpi=160, bbox_inches="tight")
plt.show()

print("\n===== 修正后结果 =====")
print("行业平均有效股票数：{:.1f}".format(industry_count.mean()))
print("行业最小有效股票数：{}".format(industry_count.min()))
print("中性化平均可用股票数：{:.1f}".format(neutralization_ready_count.mean()))
print("中性化最小可用股票数：{}".format(neutralization_ready_count.min()))
print("平均一级行业组数：{:.1f}".format(industry_group_count.mean()))

print("\n已保存：", coverage_path)
print("已保存：", industry_coverage_path)
print("已保存：", query_log_path)
print("已保存：", metadata_path)
print("已保存：", figure_path)

In [ ]:
# ============================================================
# 01.5 稳定去杠杆 ROE：去极值、行业/市值中性化、标准化
# 不保存原始因子值；只保存可开源的质量审计结果
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------- 1. 前置变量检查 ----------
required_variables = [
    "stock_list",
    "signal_dates",
    "research_mask",
    "stable_delevered_roe_raw",
    "log_market_cap_raw",
    "industry_panel",
]
missing_variables = [
    name for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "请先运行前面的单元格，缺少变量："
        + ", ".join(missing_variables)
    )

# ---------- 2. 项目目录 ----------
PROJECT_NAME = "a-share-profitability-factor-replication"
project_candidates = [
    path
    for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if path.name == PROJECT_NAME
]
PROJECT_DIR = project_candidates[-1]

TABLE_DIR = PROJECT_DIR / "artifacts" / "tables"
META_DIR = PROJECT_DIR / "artifacts" / "metadata"
FIG_DIR = PROJECT_DIR / "artifacts" / "figures"

for directory in [TABLE_DIR, META_DIR, FIG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

signal_dates = pd.DatetimeIndex(pd.to_datetime(signal_dates))

# ---------- 3. 工具函数 ----------
MAD_N = 3.5
MIN_OBSERVATIONS = 100
MIN_INDUSTRY_GROUPS = 5


def mad_winsorize(series, n=MAD_N):
    """MAD 法去极值；不改变缺失值。"""
    clean_series = series.replace([np.inf, -np.inf], np.nan).dropna()

    if len(clean_series) == 0:
        return clean_series

    median_value = clean_series.median()
    mad_value = (clean_series - median_value).abs().median()

    if (not np.isfinite(mad_value)) or mad_value == 0:
        return clean_series

    lower_bound = median_value - n * 1.4826 * mad_value
    upper_bound = median_value + n * 1.4826 * mad_value

    return clean_series.clip(lower=lower_bound, upper=upper_bound)


def zscore(series):
    """横截面标准化，使均值为 0、标准差为 1。"""
    std_value = series.std(ddof=0)

    if (not np.isfinite(std_value)) or std_value == 0:
        return pd.Series(np.nan, index=series.index, dtype=float)

    return (series - series.mean()) / std_value


def industry_mean_dispersion(factor_series, industry_series):
    """计算各行业因子均值的标准差，用于检查行业暴露。"""
    tmp = pd.DataFrame({
        "factor": factor_series,
        "industry": industry_series,
    }).dropna()

    if tmp["industry"].nunique() < 2:
        return np.nan

    industry_means = tmp.groupby("industry")["factor"].mean()
    return industry_means.std(ddof=0)


def neutralize_one_cross_section(
    raw_factor,
    log_market_cap,
    industry,
):
    """
    单期处理：
    1. MAD 去极值
    2. 因子 ~ 常数项 + log(市值) + 行业虚拟变量
    3. 取回归残差
    4. 残差标准化
    """

    data = pd.DataFrame({
        "raw_factor": raw_factor,
        "log_market_cap": log_market_cap,
        "industry": industry,
    }).replace([np.inf, -np.inf], np.nan).dropna()

    if len(data) < MIN_OBSERVATIONS:
        raise ValueError(
            "有效股票数不足 {}，当前仅 {} 只。".format(
                MIN_OBSERVATIONS,
                len(data),
            )
        )

    if data["industry"].nunique() < MIN_INDUSTRY_GROUPS:
        raise ValueError(
            "有效行业组不足 {}，当前仅 {} 组。".format(
                MIN_INDUSTRY_GROUPS,
                data["industry"].nunique(),
            )
        )

    # 先仅对因子做去极值；市值本身已经取对数
    factor_winsorized = mad_winsorize(data["raw_factor"])
    data = data.loc[factor_winsorized.index].copy()
    data["factor_winsorized"] = factor_winsorized

    # 行业虚拟变量去掉一列，避免与常数项完全共线
    industry_dummies = pd.get_dummies(
        data["industry"],
        prefix="industry",
    ).astype(float)

    industry_dummies = industry_dummies.iloc[:, 1:]

    x_matrix = np.column_stack([
        np.ones(len(data)),
        data["log_market_cap"].to_numpy(dtype=float),
        industry_dummies.to_numpy(dtype=float),
    ])
    y_vector = data["factor_winsorized"].to_numpy(dtype=float)

    beta, _, matrix_rank, _ = np.linalg.lstsq(
        x_matrix,
        y_vector,
        rcond=None,
    )

    fitted_value = x_matrix @ beta
    residual = pd.Series(
        y_vector - fitted_value,
        index=data.index,
        name="stable_delevered_roe_neutralized",
    )

    neutralized_factor = zscore(residual)

    # 审计指标：中性化前后与市值、行业暴露的关系
    raw_cap_corr = data["factor_winsorized"].corr(data["log_market_cap"])
    neutral_cap_corr = neutralized_factor.corr(
        data.loc[neutralized_factor.index, "log_market_cap"]
    )

    raw_industry_dispersion = industry_mean_dispersion(
        data["factor_winsorized"],
        data["industry"],
    )
    neutral_industry_dispersion = industry_mean_dispersion(
        neutralized_factor,
        data.loc[neutralized_factor.index, "industry"],
    )

    total_ss = np.sum(
        (y_vector - np.mean(y_vector)) ** 2
    )
    residual_ss = np.sum((y_vector - fitted_value) ** 2)
    regression_r2 = (
        1 - residual_ss / total_ss
        if total_ss > 0 else np.nan
    )

    audit = {
        "valid_stocks": len(data),
        "industry_groups": int(data["industry"].nunique()),
        "design_matrix_columns": int(x_matrix.shape[1]),
        "matrix_rank": int(matrix_rank),
        "market_cap_beta": float(beta[1]),
        "regression_r2": float(regression_r2),
        "raw_log_market_cap_corr": float(raw_cap_corr),
        "neutralized_log_market_cap_corr": float(neutral_cap_corr),
        "raw_industry_mean_std": float(raw_industry_dispersion),
        "neutralized_industry_mean_std": float(
            neutral_industry_dispersion
        ),
    }

    return neutralized_factor, audit


# ---------- 4. 对每个信号日独立中性化 ----------
stable_delevered_roe_neutralized = pd.DataFrame(
    np.nan,
    index=signal_dates,
    columns=stock_list,
    dtype=float,
)

audit_records = []

for sequence, as_of_date in enumerate(signal_dates, start=1):
    factor_result, audit_result = neutralize_one_cross_section(
        raw_factor=stable_delevered_roe_raw.loc[as_of_date],
        log_market_cap=log_market_cap_raw.loc[as_of_date],
        industry=industry_panel.loc[as_of_date],
    )

    stable_delevered_roe_neutralized.loc[
        as_of_date,
        factor_result.index,
    ] = factor_result.values

    audit_result["signal_date"] = as_of_date
    audit_records.append(audit_result)

# 后续因子检验将使用这个变量
stable_delevered_roe_final = stable_delevered_roe_neutralized.copy()

neutralization_audit = pd.DataFrame(audit_records).set_index(
    "signal_date"
)
neutralization_audit.index.name = "signal_date"

# ---------- 5. 保存可开源审计结果 ----------
audit_path = (
    TABLE_DIR
    / "01_stable_delevered_roe_neutralization_audit.csv"
)
neutralization_audit.to_csv(
    audit_path,
    encoding="utf-8-sig",
)

metadata = {
    "factor_name": "stable_delevered_roe",
    "treatment_order": [
        "Research-universe mask at signal date T",
        "MAD winsorization with n=3.5",
        "OLS residualization against intercept, log market cap, and CITIC 2019 first-level industry dummies",
        "Cross-sectional z-score of residuals",
    ],
    "minimum_observations": MIN_OBSERVATIONS,
    "minimum_industry_groups": MIN_INDUSTRY_GROUPS,
    "industry_source": "citics_2019",
    "no_future_data_rule": (
        "Factor, market cap, industry classification and universe "
        "are all aligned to the monthly signal date T."
    ),
    "raw_factor_values_saved_to_repository": False,
}
metadata_path = META_DIR / "01_neutralization_rules.json"
with open(metadata_path, "w", encoding="utf-8") as file:
    json.dump(metadata, file, ensure_ascii=False, indent=2)

# ---------- 6. 可视化中性化效果 ----------
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].plot(
    neutralization_audit.index,
    neutralization_audit["raw_log_market_cap_corr"],
    label="Before neutralization",
    linewidth=1.5,
)
axes[0].plot(
    neutralization_audit.index,
    neutralization_audit["neutralized_log_market_cap_corr"],
    label="After neutralization",
    linewidth=1.5,
)
axes[0].axhline(0, color="gray", linestyle="--", linewidth=1)
axes[0].set_title("Correlation with Log Market Cap")
axes[0].set_xlabel("Date")
axes[0].set_ylabel("Cross-sectional Correlation")
axes[0].grid(alpha=0.25)
axes[0].legend()

axes[1].plot(
    neutralization_audit.index,
    neutralization_audit["raw_industry_mean_std"],
    label="Before neutralization",
    linewidth=1.5,
)
axes[1].plot(
    neutralization_audit.index,
    neutralization_audit["neutralized_industry_mean_std"],
    label="After neutralization",
    linewidth=1.5,
)
axes[1].set_title("Dispersion of Industry Mean Exposure")
axes[1].set_xlabel("Date")
axes[1].set_ylabel("Std of Industry Mean Factor Values")
axes[1].grid(alpha=0.25)
axes[1].legend()

plt.tight_layout()

figure_path = FIG_DIR / "01_neutralization_quality.png"
plt.savefig(figure_path, dpi=160, bbox_inches="tight")
plt.show()

# ---------- 7. 输出检查 ----------
final_valid_count = stable_delevered_roe_final.notna().sum(axis=1)

print("中性化后平均有效股票数：{:.1f}".format(final_valid_count.mean()))
print("中性化后最小有效股票数：{}".format(final_valid_count.min()))
print(
    "市值相关系数绝对值：中性化前均值 {:.4f}，中性化后均值 {:.6f}".format(
        neutralization_audit["raw_log_market_cap_corr"].abs().mean(),
        neutralization_audit[
            "neutralized_log_market_cap_corr"
        ].abs().mean(),
    )
)
print(
    "行业均值离散度：中性化前均值 {:.4f}，中性化后均值 {:.6f}".format(
        neutralization_audit["raw_industry_mean_std"].mean(),
        neutralization_audit[
            "neutralized_industry_mean_std"
        ].mean(),
    )
)
print("已保存：", audit_path)
print("已保存：", metadata_path)
print("已保存：", figure_path)

In [ ]:
# ============================================================
# 02.1 构建未来 20 个交易日收益标签
# 信号 T -> T+1 开盘买入 -> T+21 开盘卖出
# 仅保存时间表与覆盖率，不保存价格和个股收益原始数据
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rqdatac import get_price, get_trading_dates

# ---------- 1. 前置变量检查 ----------
required_variables = [
    "stock_list",
    "signal_dates",
    "stable_delevered_roe_final",
]
missing_variables = [
    name for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "请先运行前面的单元格，缺少变量："
        + ", ".join(missing_variables)
    )

# ---------- 2. 项目目录 ----------
PROJECT_NAME = "a-share-profitability-factor-replication"
project_candidates = [
    path
    for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if path.name == PROJECT_NAME
]
PROJECT_DIR = project_candidates[-1]

TABLE_DIR = PROJECT_DIR / "artifacts" / "tables"
META_DIR = PROJECT_DIR / "artifacts" / "metadata"
FIG_DIR = PROJECT_DIR / "artifacts" / "figures"

for directory in [TABLE_DIR, META_DIR, FIG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# ---------- 3. 收益标签规则 ----------
RETURN_HORIZON = 20
RETURN_CALENDAR_END = "2026-03-31"

signal_dates = pd.DatetimeIndex(pd.to_datetime(signal_dates))

# 为最后一个 2025 年末信号补足未来 20 个交易日价格
trading_calendar = pd.DatetimeIndex(
    pd.to_datetime(
        get_trading_dates(
            signal_dates.min(),
            RETURN_CALENDAR_END,
        )
    )
)

# ---------- 4. 建立 T / T+1 / T+21 时间表 ----------
schedule_records = []

for signal_date in signal_dates:
    signal_position = trading_calendar.get_loc(signal_date)

    entry_position = signal_position + 1
    exit_position = entry_position + RETURN_HORIZON

    if exit_position >= len(trading_calendar):
        entry_date = pd.NaT
        exit_date = pd.NaT
        status = "insufficient_future_calendar"
    else:
        entry_date = trading_calendar[entry_position]
        exit_date = trading_calendar[exit_position]
        status = "ok"

    schedule_records.append({
        "signal_date": signal_date,
        "entry_date_t_plus_1_open": entry_date,
        "exit_date_t_plus_21_open": exit_date,
        "holding_intervals": RETURN_HORIZON,
        "status": status,
    })

return_schedule = pd.DataFrame(schedule_records)

if (return_schedule["status"] != "ok").any():
    raise RuntimeError(
        "最后若干信号日没有足够未来交易日，"
        "请将 RETURN_CALENDAR_END 调整得更晚。"
    )

# ---------- 5. 一次性提取开盘价 ----------
first_entry_date = return_schedule[
    "entry_date_t_plus_1_open"
].min()

last_exit_date = return_schedule[
    "exit_date_t_plus_21_open"
].max()

open_price_long = get_price(
    order_book_ids=stock_list,
    start_date=first_entry_date,
    end_date=last_exit_date,
    frequency="1d",
    fields="open",
    adjust_type="pre",
)

open_price = (
    open_price_long["open"]
    .unstack("order_book_id")
)

open_price.index = pd.to_datetime(open_price.index)
open_price = open_price.reindex(columns=stock_list)

# ---------- 6. 计算未来收益 ----------
forward_return_20d = pd.DataFrame(
    np.nan,
    index=signal_dates,
    columns=stock_list,
    dtype=float,
)

for _, row in return_schedule.iterrows():
    signal_date = pd.Timestamp(row["signal_date"])
    entry_date = pd.Timestamp(row["entry_date_t_plus_1_open"])
    exit_date = pd.Timestamp(row["exit_date_t_plus_21_open"])

    entry_open = open_price.loc[entry_date]
    exit_open = open_price.loc[exit_date]

    # 价格为缺失或非正时无法形成有效收益标签
    valid_price_mask = (
        entry_open.notna()
        & exit_open.notna()
        & (entry_open > 0)
        & (exit_open > 0)
    )

    one_period_return = (
        exit_open / entry_open - 1
    ).where(valid_price_mask)

    forward_return_20d.loc[signal_date] = one_period_return

# ---------- 7. 覆盖率审计 ----------
factor_valid_count = stable_delevered_roe_final.notna().sum(axis=1)
forward_return_valid_count = forward_return_20d.notna().sum(axis=1)

factor_return_pair_mask = (
    stable_delevered_roe_final.notna()
    & forward_return_20d.notna()
)
factor_return_pair_count = factor_return_pair_mask.sum(axis=1)

return_coverage = (
    return_schedule
    .set_index("signal_date")
    .reindex(signal_dates)
    .copy()
)

return_coverage["factor_valid"] = factor_valid_count
return_coverage["forward_return_valid"] = forward_return_valid_count
return_coverage["factor_return_paired"] = factor_return_pair_count
return_coverage.index.name = "signal_date"

# ---------- 8. 保存可开源结果 ----------
schedule_path = TABLE_DIR / "02_return_schedule.csv"
return_schedule.to_csv(
    schedule_path,
    index=False,
    encoding="utf-8-sig",
)

coverage_path = TABLE_DIR / "02_forward_return_20d_coverage.csv"
return_coverage.to_csv(
    coverage_path,
    encoding="utf-8-sig",
)

metadata = {
    "return_name": "forward_return_20d",
    "entry_rule": "Buy at the open of the first trading day after signal date T",
    "exit_rule": "Sell at the open of the 21st trading day after signal date T",
    "holding_intervals": RETURN_HORIZON,
    "price_field": "open",
    "price_adjustment": "pre-adjusted",
    "signal_timing_rule": (
        "The factor is constructed only from information available "
        "at signal date T. Future price is label data only."
    ),
    "execution_constraints_included": False,
    "execution_constraint_note": (
        "Suspension, limit-up and tradability constraints will be "
        "introduced in the later strategy backtest, not in this first IC label."
    ),
    "raw_price_or_return_panel_saved_to_repository": False,
}
metadata_path = META_DIR / "02_forward_return_20d_rules.json"
with open(metadata_path, "w", encoding="utf-8") as file:
    json.dump(metadata, file, ensure_ascii=False, indent=2)

# ---------- 9. 保存覆盖率图 ----------
plt.figure(figsize=(12, 5))

plt.plot(
    return_coverage.index,
    return_coverage["factor_valid"],
    label="factor_valid",
    linewidth=1.5,
)
plt.plot(
    return_coverage.index,
    return_coverage["forward_return_valid"],
    label="forward_return_valid",
    linewidth=1.5,
)
plt.plot(
    return_coverage.index,
    return_coverage["factor_return_paired"],
    label="factor_return_paired",
    linewidth=1.8,
)

plt.title("Forward 20-Day Return Label Coverage")
plt.xlabel("Signal Date")
plt.ylabel("Stock Count")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()

figure_path = FIG_DIR / "02_forward_return_20d_coverage.png"
plt.savefig(figure_path, dpi=160, bbox_inches="tight")
plt.show()

# ---------- 10. 输出检查 ----------
print("信号日期数：{}".format(len(return_schedule)))
print(
    "收益标签区间：{} 至 {}".format(
        first_entry_date.strftime("%Y-%m-%d"),
        last_exit_date.strftime("%Y-%m-%d"),
    )
)
print(
    "因子与收益平均配对股票数：{:.1f}".format(
        factor_return_pair_count.mean()
    )
)
print(
    "因子与收益最小配对股票数：{}".format(
        factor_return_pair_count.min()
    )
)
print("已保存：", schedule_path)
print("已保存：", coverage_path)
print("已保存：", metadata_path)
print("已保存：", figure_path)

In [ ]:
# ============================================================
# 02.1A 修正收益标签覆盖率展示口径
# 不重算价格、不重算收益，只修正审计表与图
# ============================================================

required_variables = [
    "forward_return_20d",
    "stable_delevered_roe_final",
    "research_mask",
    "return_schedule",
    "signal_dates",
]
missing_variables = [
    name for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "缺少变量：" + ", ".join(missing_variables)
    )

signal_dates = pd.DatetimeIndex(pd.to_datetime(signal_dates))
research_mask = research_mask.reindex(
    index=signal_dates,
    columns=stock_list,
).fillna(False)

# 全历史股票并集中的可用收益数量：仅作数据质量参考
return_valid_all_stock_list = forward_return_20d.notna().sum(axis=1)

# 当前月研究券池中的可用收益数量：项目展示使用这个指标
return_valid_in_research_universe = (
    forward_return_20d.notna()
    & research_mask
).sum(axis=1)

factor_valid_count = stable_delevered_roe_final.notna().sum(axis=1)

factor_return_pair_count = (
    stable_delevered_roe_final.notna()
    & forward_return_20d.notna()
).sum(axis=1)

# 覆盖此前口径不够清晰的文件
return_coverage = (
    return_schedule
    .set_index("signal_date")
    .reindex(signal_dates)
    .copy()
)

return_coverage["research_universe_stocks"] = research_mask.sum(axis=1)
return_coverage["factor_valid"] = factor_valid_count
return_coverage[
    "forward_return_valid_all_stock_list"
] = return_valid_all_stock_list
return_coverage[
    "forward_return_valid_in_research_universe"
] = return_valid_in_research_universe
return_coverage["factor_return_paired"] = factor_return_pair_count
return_coverage.index.name = "signal_date"

coverage_path = TABLE_DIR / "02_forward_return_20d_coverage.csv"
return_coverage.to_csv(
    coverage_path,
    encoding="utf-8-sig",
)

# 更新规则说明
metadata_path = META_DIR / "02_forward_return_20d_rules.json"

with open(metadata_path, "r", encoding="utf-8") as file:
    return_metadata = json.load(file)

return_metadata["coverage_metric_note"] = {
    "forward_return_valid_all_stock_list": (
        "Valid price labels across the full historical union of CSI 800 constituents."
    ),
    "forward_return_valid_in_research_universe": (
        "Valid price labels among stocks in the signal-date research universe."
    ),
    "factor_return_paired": (
        "Stocks with both a final factor value and a valid future return label."
    ),
}

with open(metadata_path, "w", encoding="utf-8") as file:
    json.dump(return_metadata, file, ensure_ascii=False, indent=2)

# 修正展示图：只画研究券池内真正相关的三项
plt.figure(figsize=(12, 5))

plt.plot(
    return_coverage.index,
    return_coverage["research_universe_stocks"],
    label="research_universe_stocks",
    linewidth=1.5,
)
plt.plot(
    return_coverage.index,
    return_coverage["factor_valid"],
    label="factor_valid",
    linewidth=1.5,
)
plt.plot(
    return_coverage.index,
    return_coverage["forward_return_valid_in_research_universe"],
    label="forward_return_valid_in_research_universe",
    linewidth=1.5,
)
plt.plot(
    return_coverage.index,
    return_coverage["factor_return_paired"],
    label="factor_return_paired",
    linewidth=1.8,
)

plt.title("Forward 20-Day Return Label Coverage in Research Universe")
plt.xlabel("Signal Date")
plt.ylabel("Stock Count")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()

figure_path = FIG_DIR / "02_forward_return_20d_coverage.png"
plt.savefig(figure_path, dpi=160, bbox_inches="tight")
plt.show()

print(
    "研究券池内未来收益平均有效数：{:.1f}".format(
        return_valid_in_research_universe.mean()
    )
)
print(
    "因子与收益平均配对数：{:.1f}".format(
        factor_return_pair_count.mean()
    )
)
print(
    "配对覆盖率：{:.2f}%".format(
        100 * factor_return_pair_count.sum() / factor_valid_count.sum()
    )
)
print("已更新：", coverage_path)
print("已更新：", metadata_path)
print("已更新：", figure_path)

In [ ]:
# ============================================================
# 02.2 稳定去杠杆 ROE 的单因子有效性检验
# IC / RankIC / 五分组 / 高低组收益
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import t as student_t

# ---------- 1. 前置变量检查 ----------
required_variables = [
    "stock_list",
    "signal_dates",
    "stable_delevered_roe_final",
    "forward_return_20d",
]
missing_variables = [
    name for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "缺少变量：" + ", ".join(missing_variables)
    )

signal_dates = pd.DatetimeIndex(pd.to_datetime(signal_dates))

# ---------- 2. 项目目录 ----------
PROJECT_NAME = "a-share-profitability-factor-replication"
project_candidates = [
    path
    for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if path.name == PROJECT_NAME
]
PROJECT_DIR = project_candidates[-1]

TABLE_DIR = PROJECT_DIR / "artifacts" / "tables"
META_DIR = PROJECT_DIR / "artifacts" / "metadata"
FIG_DIR = PROJECT_DIR / "artifacts" / "figures"

for directory in [TABLE_DIR, META_DIR, FIG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# ---------- 3. 每期 IC、RankIC 与五分组 ----------
GROUP_COUNT = 5

ic_records = []
quintile_records = []

for as_of_date in signal_dates:
    test_data = pd.DataFrame({
        "factor": stable_delevered_roe_final.loc[as_of_date],
        "forward_return": forward_return_20d.loc[as_of_date],
    }).replace([np.inf, -np.inf], np.nan).dropna()

    if len(test_data) < GROUP_COUNT * 20:
        raise ValueError(
            "{} 的有效样本数不足，当前为 {}。".format(
                as_of_date.strftime("%Y-%m-%d"),
                len(test_data),
            )
        )

    # Pearson IC 与 Spearman RankIC
    ic_value = test_data["factor"].corr(
        test_data["forward_return"],
        method="pearson",
    )
    rank_ic_value = test_data["factor"].corr(
        test_data["forward_return"],
        method="spearman",
    )

    ic_records.append({
        "signal_date": as_of_date,
        "valid_stocks": len(test_data),
        "ic": ic_value,
        "rank_ic": rank_ic_value,
    })

    # 用排序值分成五组；Q1 为最低因子组，Q5 为最高因子组
    test_data["quintile"] = (
        pd.qcut(
            test_data["factor"].rank(method="first"),
            q=GROUP_COUNT,
            labels=False,
        ) + 1
    )

    group_return = test_data.groupby(
        "quintile"
    )["forward_return"].mean()

    group_count = test_data.groupby(
        "quintile"
    )["forward_return"].count()

    quintile_record = {
        "signal_date": as_of_date,
        "q1_return": group_return.get(1, np.nan),
        "q2_return": group_return.get(2, np.nan),
        "q3_return": group_return.get(3, np.nan),
        "q4_return": group_return.get(4, np.nan),
        "q5_return": group_return.get(5, np.nan),
        "high_minus_low_return": (
            group_return.get(5, np.nan)
            - group_return.get(1, np.nan)
        ),
        "q1_count": group_count.get(1, np.nan),
        "q2_count": group_count.get(2, np.nan),
        "q3_count": group_count.get(3, np.nan),
        "q4_count": group_count.get(4, np.nan),
        "q5_count": group_count.get(5, np.nan),
    }
    quintile_records.append(quintile_record)

ic_timeseries = pd.DataFrame(ic_records).set_index("signal_date")
ic_timeseries.index.name = "signal_date"

quintile_timeseries = pd.DataFrame(
    quintile_records
).set_index("signal_date")
quintile_timeseries.index.name = "signal_date"

# ---------- 4. 各样本区间汇总 ----------
def series_t_stat(series):
    series = series.dropna()

    if len(series) < 2 or series.std(ddof=1) == 0:
        return np.nan, np.nan

    t_stat = (
        series.mean()
        / (series.std(ddof=1) / np.sqrt(len(series)))
    )
    p_value = 2 * student_t.sf(
        np.abs(t_stat),
        df=len(series) - 1,
    )
    return t_stat, p_value


def summarize_period(period_name, date_mask):
    period_ic = ic_timeseries.loc[date_mask]
    period_group = quintile_timeseries.loc[date_mask]

    ic_t_stat, ic_p_value = series_t_stat(period_ic["ic"])
    rank_ic_t_stat, rank_ic_p_value = series_t_stat(
        period_ic["rank_ic"]
    )
    hml_t_stat, hml_p_value = series_t_stat(
        period_group["high_minus_low_return"]
    )

    return {
        "period": period_name,
        "observations": len(period_ic),
        "average_valid_stocks": period_ic["valid_stocks"].mean(),
        "mean_ic": period_ic["ic"].mean(),
        "ic_std": period_ic["ic"].std(ddof=1),
        "icir_annualized": (
            period_ic["ic"].mean()
            / period_ic["ic"].std(ddof=1)
            * np.sqrt(12)
        ),
        "ic_t_stat": ic_t_stat,
        "ic_p_value": ic_p_value,
        "mean_rank_ic": period_ic["rank_ic"].mean(),
        "rank_ic_std": period_ic["rank_ic"].std(ddof=1),
        "rank_icir_annualized": (
            period_ic["rank_ic"].mean()
            / period_ic["rank_ic"].std(ddof=1)
            * np.sqrt(12)
        ),
        "rank_ic_t_stat": rank_ic_t_stat,
        "rank_ic_p_value": rank_ic_p_value,
        "positive_rank_ic_ratio": (
            period_ic["rank_ic"] > 0
        ).mean(),
        "mean_q1_return": period_group["q1_return"].mean(),
        "mean_q2_return": period_group["q2_return"].mean(),
        "mean_q3_return": period_group["q3_return"].mean(),
        "mean_q4_return": period_group["q4_return"].mean(),
        "mean_q5_return": period_group["q5_return"].mean(),
        "mean_high_minus_low_return": (
            period_group["high_minus_low_return"].mean()
        ),
        "high_minus_low_t_stat": hml_t_stat,
        "high_minus_low_p_value": hml_p_value,
        "high_minus_low_positive_ratio": (
            period_group["high_minus_low_return"] > 0
        ).mean(),
    }


in_sample_mask = (
    ic_timeseries.index
    <= pd.Timestamp("2022-12-31")
)

out_of_sample_mask = (
    ic_timeseries.index
    >= pd.Timestamp("2023-01-01")
)

summary_records = [
    summarize_period(
        "full_sample_2016_2025",
        pd.Series(True, index=ic_timeseries.index),
    ),
    summarize_period(
        "in_sample_2016_2022",
        in_sample_mask,
    ),
    summarize_period(
        "out_of_sample_2023_2025",
        out_of_sample_mask,
    ),
]

factor_effectiveness_summary = pd.DataFrame(summary_records)

# ---------- 5. 计算五分组净值，仅用于展示 ----------
quintile_return_columns = [
    "q1_return",
    "q2_return",
    "q3_return",
    "q4_return",
    "q5_return",
]

quintile_nav = (
    1 + quintile_timeseries[quintile_return_columns]
).cumprod()

high_minus_low_nav = (
    1 + quintile_timeseries["high_minus_low_return"]
).cumprod()

# ---------- 6. 保存可开源聚合结果 ----------
ic_path = TABLE_DIR / "02_ic_rankic_timeseries.csv"
ic_timeseries.to_csv(ic_path, encoding="utf-8-sig")

quintile_path = TABLE_DIR / "02_quintile_return_timeseries.csv"
quintile_timeseries.to_csv(
    quintile_path,
    encoding="utf-8-sig",
)

summary_path = TABLE_DIR / "02_factor_effectiveness_summary.csv"
factor_effectiveness_summary.to_csv(
    summary_path,
    index=False,
    encoding="utf-8-sig",
)

metadata = {
    "factor_name": "stable_delevered_roe",
    "return_label": "forward_return_20d",
    "ic": "Pearson correlation between factor at T and future 20-trading-day return",
    "rank_ic": "Spearman correlation between factor at T and future 20-trading-day return",
    "grouping": "Equal-count quintiles; Q1 is lowest factor and Q5 is highest factor",
    "portfolio_return": "Equal-weighted gross return within each quintile",
    "high_minus_low": "Q5 return minus Q1 return",
    "in_sample_period": "2016-01-01 to 2022-12-31",
    "out_of_sample_period": "2023-01-01 to 2025-12-31",
    "transaction_costs_included": False,
    "execution_constraints_included": False,
    "raw_factor_or_stock_level_return_saved_to_repository": False,
}
metadata_path = META_DIR / "02_factor_test_rules.json"
with open(metadata_path, "w", encoding="utf-8") as file:
    json.dump(metadata, file, ensure_ascii=False, indent=2)

# ---------- 7. 画图 ----------
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].plot(
    ic_timeseries.index,
    ic_timeseries["ic"],
    label="IC",
    linewidth=1.3,
)
axes[0].plot(
    ic_timeseries.index,
    ic_timeseries["rank_ic"],
    label="RankIC",
    linewidth=1.3,
)
axes[0].axhline(0, color="gray", linestyle="--", linewidth=1)
axes[0].set_title("Monthly IC and RankIC")
axes[0].set_xlabel("Signal Date")
axes[0].set_ylabel("Correlation")
axes[0].grid(alpha=0.25)
axes[0].legend()

for column in quintile_nav.columns:
    axes[1].plot(
        quintile_nav.index,
        quintile_nav[column],
        label=column.replace("_return", "").upper(),
        linewidth=1.4,
    )

axes[1].set_title("Equal-Weighted Quintile Net Value")
axes[1].set_xlabel("Signal Date")
axes[1].set_ylabel("Gross Net Value")
axes[1].grid(alpha=0.25)
axes[1].legend()

plt.tight_layout()

figure_path = FIG_DIR / "02_factor_effectiveness.png"
plt.savefig(figure_path, dpi=160, bbox_inches="tight")
plt.show()

# ---------- 8. 控制台关键结果 ----------
display_columns = [
    "period",
    "observations",
    "mean_ic",
    "icir_annualized",
    "mean_rank_ic",
    "rank_icir_annualized",
    "positive_rank_ic_ratio",
    "mean_q1_return",
    "mean_q5_return",
    "mean_high_minus_low_return",
    "high_minus_low_t_stat",
]

print(
    factor_effectiveness_summary[
        display_columns
    ].round(4).to_string(index=False)
)

print("\n已保存：", ic_path)
print("已保存：", quintile_path)
print("已保存：", summary_path)
print("已保存：", metadata_path)
print("已保存：", figure_path)

In [ ]:
# ============================================================
# 02.3 盈利能力因子组件比较
# 固定规则比较：原始 ROE / 去杠杆 ROE / 稳定性 / 稳定去杠杆 ROE
# 不做参数寻优，不按全样本结果调参
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import t as student_t

# ---------- 1. 前置变量检查 ----------
required_variables = [
    "stock_list",
    "signal_dates",
    "roe_ttm_baseline_raw",
    "delevered_roe_raw",
    "roe_volatility_raw",
    "stable_delevered_roe_raw",
    "stable_delevered_roe_final",
    "log_market_cap_raw",
    "industry_panel",
    "forward_return_20d",
    "neutralize_one_cross_section",
]

missing_variables = [
    name for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "缺少变量：" + ", ".join(missing_variables)
    )

signal_dates = pd.DatetimeIndex(pd.to_datetime(signal_dates))

# ---------- 2. 项目目录 ----------
PROJECT_NAME = "a-share-profitability-factor-replication"
project_candidates = [
    path
    for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if path.name == PROJECT_NAME
]
PROJECT_DIR = project_candidates[-1]

TABLE_DIR = PROJECT_DIR / "artifacts" / "tables"
META_DIR = PROJECT_DIR / "artifacts" / "metadata"
FIG_DIR = PROJECT_DIR / "artifacts" / "figures"

for directory in [TABLE_DIR, META_DIR, FIG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# ---------- 3. 固定的四种对比口径 ----------
# 注意：负 ROE 波动率越高，表示盈利越稳定
raw_variant_panels = {
    "roe_ttm_baseline": roe_ttm_baseline_raw,
    "delevered_roe": delevered_roe_raw,
    "negative_roe_volatility": -roe_volatility_raw,
    "stable_delevered_roe": stable_delevered_roe_raw,
}

variant_factor_panels = {}
variant_coverage_records = []

# 对所有变体执行完全一致的中性化规则
for variant_name, raw_panel in raw_variant_panels.items():
    final_panel = pd.DataFrame(
        np.nan,
        index=signal_dates,
        columns=stock_list,
        dtype=float,
    )

    for as_of_date in signal_dates:
        neutralized_factor, audit = neutralize_one_cross_section(
            raw_factor=raw_panel.loc[as_of_date],
            log_market_cap=log_market_cap_raw.loc[as_of_date],
            industry=industry_panel.loc[as_of_date],
        )

        final_panel.loc[
            as_of_date,
            neutralized_factor.index,
        ] = neutralized_factor.values

        variant_coverage_records.append({
            "signal_date": as_of_date,
            "factor_variant": variant_name,
            "valid_stocks": audit["valid_stocks"],
        })

    variant_factor_panels[variant_name] = final_panel

# 检查主因子重建是否与此前结果一致
stable_factor_max_difference = (
    variant_factor_panels["stable_delevered_roe"]
    - stable_delevered_roe_final
).abs().max().max()

# ---------- 4. 统计函数 ----------
def get_t_stat(series):
    series = series.dropna()

    if len(series) < 2 or series.std(ddof=1) == 0:
        return np.nan, np.nan

    t_stat = (
        series.mean()
        / (series.std(ddof=1) / np.sqrt(len(series)))
    )
    p_value = 2 * student_t.sf(
        np.abs(t_stat),
        df=len(series) - 1,
    )
    return t_stat, p_value


def evaluate_factor_variant(factor_name, factor_panel):
    metric_records = []
    hml_records = []

    for as_of_date in signal_dates:
        data = pd.DataFrame({
            "factor": factor_panel.loc[as_of_date],
            "forward_return": forward_return_20d.loc[as_of_date],
        }).replace([np.inf, -np.inf], np.nan).dropna()

        if len(data) < 100:
            raise ValueError(
                "{} 在 {} 的样本不足 100 只。".format(
                    factor_name,
                    as_of_date.strftime("%Y-%m-%d"),
                )
            )

        ic_value = data["factor"].corr(
            data["forward_return"],
            method="pearson",
        )
        rank_ic_value = data["factor"].corr(
            data["forward_return"],
            method="spearman",
        )

        data["quintile"] = (
            pd.qcut(
                data["factor"].rank(method="first"),
                q=5,
                labels=False,
            ) + 1
        )

        grouped_return = data.groupby(
            "quintile"
        )["forward_return"].mean()

        metric_records.append({
            "signal_date": as_of_date,
            "factor_variant": factor_name,
            "valid_stocks": len(data),
            "ic": ic_value,
            "rank_ic": rank_ic_value,
        })

        hml_records.append({
            "signal_date": as_of_date,
            "factor_variant": factor_name,
            "q1_return": grouped_return.get(1, np.nan),
            "q5_return": grouped_return.get(5, np.nan),
            "high_minus_low_return": (
                grouped_return.get(5, np.nan)
                - grouped_return.get(1, np.nan)
            ),
        })

    return (
        pd.DataFrame(metric_records),
        pd.DataFrame(hml_records),
    )


# ---------- 5. 逐个变体检验 ----------
all_metric_records = []
all_hml_records = []

for variant_name, factor_panel in variant_factor_panels.items():
    variant_metric, variant_hml = evaluate_factor_variant(
        variant_name,
        factor_panel,
    )
    all_metric_records.append(variant_metric)
    all_hml_records.append(variant_hml)

variant_metrics = pd.concat(
    all_metric_records,
    ignore_index=True,
)

variant_hml = pd.concat(
    all_hml_records,
    ignore_index=True,
)

# ---------- 6. 全样本 / 样本内 / 样本外汇总 ----------
def summarize_variant_period(
    factor_name,
    period_name,
    start_date,
    end_date,
):
    factor_metrics = variant_metrics[
        (variant_metrics["factor_variant"] == factor_name)
        & (variant_metrics["signal_date"] >= start_date)
        & (variant_metrics["signal_date"] <= end_date)
    ]

    factor_hml = variant_hml[
        (variant_hml["factor_variant"] == factor_name)
        & (variant_hml["signal_date"] >= start_date)
        & (variant_hml["signal_date"] <= end_date)
    ]

    ic_t_stat, ic_p_value = get_t_stat(factor_metrics["ic"])
    rank_ic_t_stat, rank_ic_p_value = get_t_stat(
        factor_metrics["rank_ic"]
    )
    hml_t_stat, hml_p_value = get_t_stat(
        factor_hml["high_minus_low_return"]
    )

    return {
        "factor_variant": factor_name,
        "period": period_name,
        "observations": len(factor_metrics),
        "average_valid_stocks": factor_metrics["valid_stocks"].mean(),
        "mean_ic": factor_metrics["ic"].mean(),
        "icir_annualized": (
            factor_metrics["ic"].mean()
            / factor_metrics["ic"].std(ddof=1)
            * np.sqrt(12)
        ),
        "ic_t_stat": ic_t_stat,
        "ic_p_value": ic_p_value,
        "mean_rank_ic": factor_metrics["rank_ic"].mean(),
        "rank_icir_annualized": (
            factor_metrics["rank_ic"].mean()
            / factor_metrics["rank_ic"].std(ddof=1)
            * np.sqrt(12)
        ),
        "rank_ic_t_stat": rank_ic_t_stat,
        "rank_ic_p_value": rank_ic_p_value,
        "positive_rank_ic_ratio": (
            factor_metrics["rank_ic"] > 0
        ).mean(),
        "mean_high_minus_low_return": (
            factor_hml["high_minus_low_return"].mean()
        ),
        "high_minus_low_t_stat": hml_t_stat,
        "high_minus_low_p_value": hml_p_value,
    }


periods = {
    "full_sample_2016_2025": (
        pd.Timestamp("2016-01-01"),
        pd.Timestamp("2025-12-31"),
    ),
    "in_sample_2016_2022": (
        pd.Timestamp("2016-01-01"),
        pd.Timestamp("2022-12-31"),
    ),
    "out_of_sample_2023_2025": (
        pd.Timestamp("2023-01-01"),
        pd.Timestamp("2025-12-31"),
    ),
}

comparison_summary_records = []

for variant_name in raw_variant_panels:
    for period_name, (start_date, end_date) in periods.items():
        comparison_summary_records.append(
            summarize_variant_period(
                variant_name,
                period_name,
                start_date,
                end_date,
            )
        )

profitability_variant_comparison = pd.DataFrame(
    comparison_summary_records
)

# ---------- 7. 保存可开源聚合结果 ----------
coverage_path = TABLE_DIR / "02_profitability_variant_coverage.csv"
pd.DataFrame(variant_coverage_records).to_csv(
    coverage_path,
    index=False,
    encoding="utf-8-sig",
)

rankic_path = TABLE_DIR / "02_profitability_variant_monthly_rankic.csv"
variant_metrics[
    [
        "signal_date",
        "factor_variant",
        "valid_stocks",
        "ic",
        "rank_ic",
    ]
].to_csv(
    rankic_path,
    index=False,
    encoding="utf-8-sig",
)

hml_path = TABLE_DIR / "02_profitability_variant_monthly_hml.csv"
variant_hml.to_csv(
    hml_path,
    index=False,
    encoding="utf-8-sig",
)

summary_path = TABLE_DIR / "02_profitability_variant_comparison.csv"
profitability_variant_comparison.to_csv(
    summary_path,
    index=False,
    encoding="utf-8-sig",
)

metadata = {
    "purpose": (
        "Component attribution of ROE improvement without "
        "parameter optimization."
    ),
    "fixed_variants": {
        "roe_ttm_baseline": "Raw TTM ROE",
        "delevered_roe": (
            "ROE residual after removing the equity multiplier"
        ),
        "negative_roe_volatility": (
            "Negative volatility of the latest nine TTM ROE observations"
        ),
        "stable_delevered_roe": (
            "Equal-weighted combination of delevered ROE and "
            "negative ROE volatility"
        ),
    },
    "common_treatment": (
        "All variants use the same MAD winsorization, CITIC 2019 "
        "first-level industry neutralization, log-market-cap "
        "neutralization, and cross-sectional z-score."
    ),
    "sample_split": {
        "in_sample": "2016-2022",
        "out_of_sample": "2023-2025",
    },
    "raw_factor_or_stock_level_return_saved_to_repository": False,
}
metadata_path = META_DIR / "02_profitability_variant_comparison_rules.json"
with open(metadata_path, "w", encoding="utf-8") as file:
    json.dump(metadata, file, ensure_ascii=False, indent=2)

# ---------- 8. 图：各变体的 RankIC ----------
plot_data = profitability_variant_comparison.pivot(
    index="factor_variant",
    columns="period",
    values="mean_rank_ic",
).reindex(raw_variant_panels.keys())

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

plot_data["full_sample_2016_2025"].plot(
    kind="bar",
    ax=axes[0],
    color="#4C78A8",
)
axes[0].axhline(0, color="gray", linestyle="--", linewidth=1)
axes[0].set_title("Full-Sample Mean RankIC")
axes[0].set_xlabel("")
axes[0].set_ylabel("Mean RankIC")
axes[0].tick_params(axis="x", rotation=25)
axes[0].grid(axis="y", alpha=0.25)

plot_data["out_of_sample_2023_2025"].plot(
    kind="bar",
    ax=axes[1],
    color="#F58518",
)
axes[1].axhline(0, color="gray", linestyle="--", linewidth=1)
axes[1].set_title("Out-of-Sample Mean RankIC")
axes[1].set_xlabel("")
axes[1].set_ylabel("Mean RankIC")
axes[1].tick_params(axis="x", rotation=25)
axes[1].grid(axis="y", alpha=0.25)

plt.tight_layout()

figure_path = FIG_DIR / "02_profitability_variant_comparison.png"
plt.savefig(figure_path, dpi=160, bbox_inches="tight")
plt.show()

# ---------- 9. 输出 ----------
display_columns = [
    "factor_variant",
    "period",
    "mean_rank_ic",
    "rank_icir_annualized",
    "positive_rank_ic_ratio",
    "mean_high_minus_low_return",
    "high_minus_low_t_stat",
]

print(
    profitability_variant_comparison[
        display_columns
    ].round(4).to_string(index=False)
)

print(
    "\n稳定去杠杆 ROE 与此前版本的最大差异：{:.12f}".format(
        stable_factor_max_difference
    )
)

print("已保存：", coverage_path)
print("已保存：", rankic_path)
print("已保存：", hml_path)
print("已保存：", summary_path)
print("已保存：", metadata_path)
print("已保存：", figure_path)

In [ ]:
# ============================================================
# 02.4 保存研究决策日志
# 记录哪些结论由样本外结果支持，防止事后调参
# ============================================================

required_variables = [
    "profitability_variant_comparison",
    "TABLE_DIR",
    "META_DIR",
]
missing_variables = [
    name for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "缺少变量：" + ", ".join(missing_variables)
    )

oos_result = profitability_variant_comparison[
    profitability_variant_comparison["period"]
    == "out_of_sample_2023_2025"
].copy()

decision_map = {
    "roe_ttm_baseline": {
        "decision": "reject_as_standalone_factor",
        "reason": (
            "Out-of-sample RankIC and high-minus-low return are both negative."
        ),
        "next_action": (
            "Keep only as a baseline comparison; do not add to factor library."
        ),
    },
    "delevered_roe": {
        "decision": "reject_as_standalone_factor",
        "reason": (
            "Out-of-sample RankIC and high-minus-low return are both negative."
        ),
        "next_action": (
            "Keep only as a baseline comparison; do not add to factor library."
        ),
    },
    "negative_roe_volatility": {
        "decision": "candidate_not_confirmed",
        "reason": (
            "Out-of-sample direction remains positive, but statistical evidence "
            "is insufficient for a standalone alpha claim."
        ),
        "next_action": (
            "Retain as a profitability-library candidate and test together with "
            "ROIC and FCFFIC; do not tune its lookback window."
        ),
    },
    "stable_delevered_roe": {
        "decision": "not_a_standalone_factor",
        "reason": (
            "Positive out-of-sample RankIC is weak and high-minus-low return "
            "is close to zero."
        ),
        "next_action": (
            "Retain for component attribution only; do not use as a standalone "
            "portfolio signal."
        ),
    },
}

decision_log = oos_result.copy()

decision_log["decision"] = decision_log[
    "factor_variant"
].map(lambda name: decision_map[name]["decision"])

decision_log["research_interpretation"] = decision_log[
    "factor_variant"
].map(lambda name: decision_map[name]["reason"])

decision_log["next_action"] = decision_log[
    "factor_variant"
].map(lambda name: decision_map[name]["next_action"])

decision_log = decision_log[
    [
        "factor_variant",
        "period",
        "observations",
        "mean_rank_ic",
        "rank_icir_annualized",
        "positive_rank_ic_ratio",
        "mean_high_minus_low_return",
        "high_minus_low_t_stat",
        "high_minus_low_p_value",
        "decision",
        "research_interpretation",
        "next_action",
    ]
]

decision_path = TABLE_DIR / "02_profitability_research_decision_log.csv"
decision_log.to_csv(
    decision_path,
    index=False,
    encoding="utf-8-sig",
)

decision_metadata = {
    "decision_date": "2026-09-28",
    "decision_principle": (
        "Do not optimize factor weights, lookback windows, or grouping rules "
        "after observing the 2016-2025 results."
    ),
    "out_of_sample_period": "2023-01-01 to 2025-12-31",
    "research_conclusion": (
        "Profitability stability is a candidate component, not a confirmed "
        "standalone alpha. Baseline ROE and delevered ROE are rejected as "
        "standalone factors in this project."
    ),
    "next_research_stage": (
        "Construct and evaluate ROIC and reconstructed FCFFIC before "
        "forming any multi-factor profitability composite."
    ),
}

metadata_path = META_DIR / "02_profitability_research_decision.json"
with open(metadata_path, "w", encoding="utf-8") as file:
    json.dump(decision_metadata, file, ensure_ascii=False, indent=2)

print(
    decision_log[
        [
            "factor_variant",
            "mean_rank_ic",
            "mean_high_minus_low_return",
            "high_minus_low_t_stat",
            "decision",
        ]
    ].round(4).to_string(index=False)
)

print("\n已保存：", decision_path)
print("已保存：", metadata_path)

In [ ]:
# ============================================================
# 02.4 恢复模式：从已保存汇总表生成研究决策日志
# 不依赖此前 Notebook 内存变量
# ============================================================

from pathlib import Path
import json
import pandas as pd

# ---------- 1. 定位项目目录 ----------
PROJECT_NAME = "a-share-profitability-factor-replication"

project_candidates = [
    path
    for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if path.name == PROJECT_NAME
]

if not project_candidates:
    raise RuntimeError("未找到项目根目录。请确认当前 Notebook 位于项目文件夹内。")

PROJECT_DIR = project_candidates[-1]
TABLE_DIR = PROJECT_DIR / "artifacts" / "tables"
META_DIR = PROJECT_DIR / "artifacts" / "metadata"

# ---------- 2. 读取已经保存的比较结果 ----------
comparison_path = (
    TABLE_DIR
    / "02_profitability_variant_comparison.csv"
)

if not comparison_path.exists():
    raise FileNotFoundError(
        "找不到已保存的比较表：" + str(comparison_path)
    )

profitability_variant_comparison = pd.read_csv(comparison_path)

# ---------- 3. 仅基于样本外结果记录研究决策 ----------
oos_result = profitability_variant_comparison[
    profitability_variant_comparison["period"]
    == "out_of_sample_2023_2025"
].copy()

decision_map = {
    "roe_ttm_baseline": {
        "decision": "reject_as_standalone_factor",
        "reason": (
            "样本外 RankIC 和高低组收益均为负，不能作为独立因子。"
        ),
        "next_action": (
            "仅保留为基准比较，不进入后续因子库。"
        ),
    },
    "delevered_roe": {
        "decision": "reject_as_standalone_factor",
        "reason": (
            "样本外 RankIC 和高低组收益均为负，不能作为独立因子。"
        ),
        "next_action": (
            "仅保留为基准比较，不进入后续因子库。"
        ),
    },
    "negative_roe_volatility": {
        "decision": "candidate_not_confirmed",
        "reason": (
            "样本外方向为正，但统计证据不足，不能称为独立 Alpha。"
        ),
        "next_action": (
            "保留为盈利能力因子库候选，与 ROIC、FCFFIC 共同检验；"
            "不再调节回看期参数。"
        ),
    },
    "stable_delevered_roe": {
        "decision": "not_a_standalone_factor",
        "reason": (
            "样本外 RankIC 偏弱，高低组收益接近于零。"
        ),
        "next_action": (
            "仅保留作成分归因，不单独构建策略。"
        ),
    },
}

decision_log = oos_result.copy()

decision_log["decision"] = decision_log[
    "factor_variant"
].map(lambda name: decision_map[name]["decision"])

decision_log["research_interpretation"] = decision_log[
    "factor_variant"
].map(lambda name: decision_map[name]["reason"])

decision_log["next_action"] = decision_log[
    "factor_variant"
].map(lambda name: decision_map[name]["next_action"])

decision_log = decision_log[
    [
        "factor_variant",
        "period",
        "observations",
        "mean_rank_ic",
        "rank_icir_annualized",
        "positive_rank_ic_ratio",
        "mean_high_minus_low_return",
        "high_minus_low_t_stat",
        "high_minus_low_p_value",
        "decision",
        "research_interpretation",
        "next_action",
    ]
]

# ---------- 4. 保存 ----------
decision_path = TABLE_DIR / "02_profitability_research_decision_log.csv"
decision_log.to_csv(
    decision_path,
    index=False,
    encoding="utf-8-sig",
)

decision_metadata = {
    "decision_principle": (
        "Do not optimize factor weights, lookback windows, or grouping rules "
        "after observing the 2016-2025 results."
    ),
    "out_of_sample_period": "2023-01-01 to 2025-12-31",
    "research_conclusion": (
        "Profitability stability is a candidate component, not a confirmed "
        "standalone alpha. Baseline ROE and delevered ROE are rejected as "
        "standalone factors in this project."
    ),
    "next_research_stage": (
        "Construct and evaluate ROIC and reconstructed FCFFIC."
    ),
}

metadata_path = META_DIR / "02_profitability_research_decision.json"
with open(metadata_path, "w", encoding="utf-8") as file:
    json.dump(decision_metadata, file, ensure_ascii=False, indent=2)

print(
    decision_log[
        [
            "factor_variant",
            "mean_rank_ic",
            "mean_high_minus_low_return",
            "high_minus_low_t_stat",
            "decision",
        ]
    ].round(4).to_string(index=False)
)

print("\n已保存：", decision_path)
print("已保存：", metadata_path)